# Clinical Trial Intelligence Platform
## End-to-End Reconciliation Demo

### Objective
Demonstrate how a new clinical subject file moves through the complete data platform:

**Amazon S3 → Bronze → Silver / Quarantine → Gold → AI/BI Dashboard**

This notebook compares the platform state before and after ingestion and traces the controlled demo records through each layer.

In [0]:
%sql
-- ============================================================
-- 1. BEFORE INGESTION — PLATFORM BASELINE
-- ============================================================

SELECT 
    'Bronze - Total Subject Rows' AS metric
    ,COUNT(*) AS record_count
FROM 
    clinical_trial_intelligence.bronze.edc_subjects

UNION ALL

SELECT 
    'Bronze - Distinct Subjects'
    ,COUNT(DISTINCT subject_id)
FROM 
    clinical_trial_intelligence.bronze.edc_subjects

UNION ALL

SELECT 
    'Silver - Current Subjects'
    ,COUNT(*)
FROM 
    clinical_trial_intelligence.silver.subjects
WHERE __END_AT IS NULL

UNION ALL

SELECT 
    'Quarantine - Subject Rows'
    ,COUNT(*)
FROM 
    clinical_trial_intelligence.quarantine.subjects

UNION ALL

SELECT 
    'Gold - Subject Summary'
    ,COUNT(*)
FROM 
    clinical_trial_intelligence.gold.gold_subject_summary

order by metric

## 2. Pre-Ingestion Demo Subject Verification

Before ingesting the controlled demo file, verify that the five demo subject IDs do not already exist in the Bronze layer.

**Expected result:** No rows returned.

In [0]:
%sql
SELECT
    subject_id
    ,study_id
    ,site_id
    ,subject_status
    ,age
    ,_source_file_name
FROM 
    clinical_trial_intelligence.bronze.edc_subjects
WHERE subject_id IN (
        '101-001-DEMO06'
        ,'101-001-DEMO07'
        ,'101-001-DEMO08'
        ,'101-001-DEMO09'
        ,'101-001-DEMO10'
    )
ORDER BY 
    subject_id;

## 3. Controlled Subject File for Ingestion

A new controlled EDC subject file will be uploaded to the Amazon S3 landing zone to demonstrate incremental ingestion and downstream data-quality processing.

**File:** `subjects_20260913.csv`

**Records:** 5 new subjects

Expected processing:

- **3 valid records** → Silver → Gold
- **2 invalid records** → Quarantine
  - `101-001-DEMO09` → `age_out_of_range`
  - `101-001-DEMO10` → `invalid_subject_status`

The file will be processed through the orchestrated workflow:

**S3 → Bronze → Silver / Quarantine → Gold → Dashboard Refresh**

### Expected Record Routing

| Subject ID | Intended Condition | Expected Destination |
|---|---|---|
| `101-001-DEMO06` | Valid enrolled subject — Age 33 | Silver → Gold |
| `101-001-DEMO07` | Valid enrolled subject — Age 46 | Silver → Gold |
| `101-001-DEMO08` | Valid screening subject — Age 31 | Silver → Gold |
| `101-001-DEMO09` | Invalid age — Age 16 | Quarantine (`age_out_of_range`) |
| `101-001-DEMO10` | Invalid status — ACTIVE | Quarantine (`invalid_subject_status`) |